In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re

# Updated Mango URL
url = "https://agmarknet.gov.in/SearchCmmMkt.aspx?Tx_Commodity=20&Tx_State=MH&Tx_District=0&Tx_Market=0&DateFrom=01-Jan-2025&DateTo=04-Nov-2025&Fr_Date=01-Jan-2025&To_Date=04-Nov-2025&Tx_Trend=2&Tx_CommodityHead=Mango&Tx_StateHead=Maharashtra&Tx_DistrictHead=--Select--&Tx_MarketHead=--Select--"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

# Fetch HTML
resp = requests.get(url, headers=headers)
print("Status:", resp.status_code)

html = resp.text

# Try reading tables directly
dfs = []
try:
    dfs = pd.read_html(html)
    print(f"Found {len(dfs)} table(s) with pandas.read_html()")
except Exception as e:
    print("pandas.read_html failed:", e)

df = None

# Locate the right table
if dfs:
    for candidate in dfs:
        cols_lower = [str(c).lower() for c in candidate.columns]
        if any("modal" in c for c in cols_lower) and any("market" in c for c in cols_lower):
            df = candidate.copy()
            break
    if df is None:
        df = max(dfs, key=lambda x: x.shape[0]*x.shape[1]).copy()

# Fallback: BeautifulSoup
if df is None:
    soup = BeautifulSoup(html, "lxml")
    table = None
    for tbl in soup.find_all("table"):
        header_text = tbl.get_text(" ").lower()
        if "modal price" in header_text and "market" in header_text:
            table = tbl
            break

    if not table:
        raise RuntimeError("❌ Could not find target table (possibly JS-rendered).")

    rows = []
    headers_row = [th.get_text(strip=True) for th in table.find_all("tr")[0].find_all(["th", "td"])]
    for tr in table.find_all("tr")[1:]:
        cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
        if cells:
            if len(cells) < len(headers_row):
                cells += [""] * (len(headers_row) - len(cells))
            rows.append(cells[:len(headers_row)])
    df = pd.DataFrame(rows, columns=headers_row)

# Clean headers
df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]

# Rename key columns for consistency
col_map = {}
for c in df.columns:
    cl = c.lower()
    if "state" in cl:
        col_map[c] = "State Name"
    elif "district" in cl:
        col_map[c] = "District Name"
    elif "market" in cl:
        col_map[c] = "Market Name"
    elif "variety" in cl:
        col_map[c] = "Variety"
    elif "group" in cl:
        col_map[c] = "Group"
    elif "arrivals" in cl:
        col_map[c] = "Arrivals (Tonnes)"
    elif "min" in cl:
        col_map[c] = "Min Price (Rs./Quintal)"
    elif "max" in cl:
        col_map[c] = "Max Price (Rs./Quintal)"
    elif "modal" in cl:
        col_map[c] = "Modal Price (Rs./Quintal)"
    elif "date" in cl:
        col_map[c] = "Reported Date"
    elif "grade" in cl:
        col_map[c] = "Grade"

df.rename(columns=col_map, inplace=True)

# Convert price and arrival columns
num_cols = ["Arrivals (Tonnes)", "Min Price (Rs./Quintal)", "Max Price (Rs./Quintal)", "Modal Price (Rs./Quintal)"]
for nc in num_cols:
    if nc in df.columns:
        df[nc] = df[nc].astype(str).str.replace(r"[^\d\.]", "", regex=True)
        df[nc] = pd.to_numeric(df[nc], errors="coerce")

# Parse dates
if "Reported Date" in df.columns:
    df["Reported Date"] = pd.to_datetime(df["Reported Date"], errors="coerce", dayfirst=True)

df.reset_index(drop=True, inplace=True)
print("✅ Final Data Shape:", df.shape)
print(df.head())

# Save (optional)
df.to_csv("agmarknet_mango_prices.csv", index=False)
print("💾 Saved to agmarknet_mango_prices.csv")

In [ ]:
df

In [ ]:
df = df.drop([1798,1799])

In [ ]:
df.info()

In [ ]:
df['Variety'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

market_avg = df.groupby('Market Name')['Modal Price (Rs./Quintal)'].mean().sort_values(ascending=False)
market_avg.head(10).plot(kind='bar', figsize=(10,5), title="Top 10 Markets by Average Mango Price")
plt.ylabel("Average Modal Price (Rs./Quintal)")
plt.show()

# Price Volatility
market_std = df.groupby('Market Name')['Modal Price (Rs./Quintal)'].std().sort_values(ascending=False)
print("Top volatile markets:\n", market_std.head(5))
